In [ ]:
#pip install geonamescache pandas numpy matplotlib

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geonamescache

In [2]:
#import danych z pakietu geonamescache

gc = geonamescache.GeonamesCache()
cities = gc.get_cities()

df = pd.DataFrame(
    [
        (v["name"], float(v["latitude"]), float(v["longitude"]), int(v["population"]))
        for v in cities.values()
        if v["countrycode"] == "PL"
    ],
    columns = ["city", "latitude", "longitude", "population"]
).sort_values("population", ascending=False).reset_index(drop=True)

In [3]:
#Utworzenie zestawów miast, na których będzie działać algorytm

small_set_20 = df.head(20)[["city", "latitude", "longitude"]].reset_index(drop=True)
medium_set_50 = df.head(50)[["city", "latitude", "longitude"]].reset_index(drop=True)
large_set_100 = df.head(100)[["city", "latitude", "longitude"]].reset_index(drop=True)

In [4]:
def haversine(lat1, long1, lat2, long2):
    """Obliczanie odległości między miastami na podstawie współrzędnych geograficznych"""
    R = 6371.0 #promień Ziemi

    """Ze względu na kulistość Ziemi, odległość między dwoma punktami liczy się inaczej, niż w przypadku odległości na płaszczyźnie.
    Jednym ze sposobów obliczania takich odległości jest użycie wzoru Haversine."""

    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(long2 - long1)

    """Mała zmiana względem typowego wzoru Haversine - używamy np.sin(dphi/2)**2 zamiast 1-np.cos(dphi) oraz tego semgo zamiennika dla dlambda,
    ponieważ sinus do kwadratu jest w stanie zapewnić większą precyzję, przy liczeniu odległości dla małych a.
    1 - np.cos(dphi) = 2 * np.sin(dphi / 2)**2 -> wyrównuje się w ostatecznym wyniku (return)."""
    
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))